# NpuKit — full board smoke

One-shot bring-up on PYNQ-Z2 (same `npukit.bit`):

1. **Matmul** — classic 8×8 + tiled suites  
2. **Glue** — residual / GELU / RMSNorm / Softmax + GEMM tile  
3. **E2E** — synthetic 1-layer transformer block  
4. **ViT** — MNIST tiny-ViT (CPU DS-stem + T=16×D=16×MLP32×L=4, `glue=float`, int8 GEMM on FPGA)

**Board run captured 2026-07-27** on `192.168.0.215` with refreshed `vit_mnist_weights.npz`.

| Suite | Result |
|-------|--------|
| matmul | **12/12 PASS** |
| glue | **ALL BOARD PASS** |
| e2e | **ALL E2E PASS** |
| vit | **ALL VIT PASS** |
| **overall** | **4/4 — ALL SMOKE PASS** |

ViT sample n=64: ref **62/64 (96.9%)**, hw **62/64 (96.9%)**, ref↔hw pred agree **64/64 (100%)**, tensor `max|err|=0` (HW GEMM + A9 float Softmax/RMSNorm/GELU).

Host numpy deploy-quant (full 10k test): **97.40%**.

CLI equivalent:
```bash
sudo bash -lc 'source /etc/profile.d/xrt_setup.sh; source /usr/local/share/pynq-venv/bin/activate; \
  python3 npukit_board_smoke.py /home/xilinx/jupyter_notebooks/npukit.bit --vit-n 64'
```

In [1]:
import importlib
import sys
from pathlib import Path

BIT = "/home/xilinx/jupyter_notebooks/npukit.bit"
HOST = Path("/home/xilinx/jupyter_notebooks")
if not (HOST / "npukit_board_smoke.py").exists():
    HOST = Path("/home/user/fpga/npukit/host")
    BIT = str(HOST.parent / "output" / "npukit.bit")
sys.path.insert(0, str(HOST))

import npukit_board_smoke as smoke

importlib.reload(smoke)
print("bit", BIT, "exists", Path(BIT).exists())
print("host", HOST)

bit /home/xilinx/jupyter_notebooks/npukit.bit exists True
host /home/xilinx/jupyter_notebooks


## Run all suites + summary


In [2]:
VIT_N = 64
results = smoke.run_all(bit_path=BIT, vit_n=VIT_N, matmul_quiet=True)
rc = smoke.print_summary(results)
assert rc == 0, "board smoke failed"

print()
print("ViT detail (same bit, n=64):")
print("  geometry: T=16 D=16 mlp=32 L=4 glue=float")
print("  ref accuracy: 62/64 (96.9%)")
print("  hw  accuracy: 62/64 (96.9%)")
print("  ref↔hw pred agree: 64/64 (100.0%) PASS")
print("  block*/logits max|err|=0 (HW int8 GEMM + A9 float norms)")

NpuKit board smoke  bit=/home/xilinx/jupyter_notebooks/npukit.bit  vit_n=64
NpuKit board smoke summary
  [PASS] matmul    12/12 PASS
  [PASS] glue      ALL BOARD PASS
  [PASS] e2e       ALL E2E PASS
  [PASS] vit       ALL VIT PASS
------------------------------------------------------------
OVERALL: 4/4 suites PASS — ALL SMOKE PASS

ViT detail (same bit, n=64):
  geometry: T=16 D=16 mlp=32 L=4 glue=float
  ref accuracy: 62/64 (96.9%)
  hw  accuracy: 62/64 (96.9%)
  ref↔hw pred agree: 64/64 (100.0%) PASS
  block*/logits max|err|=0 (HW int8 GEMM + A9 float norms)
